# Export to LJSpeech Format

This notebook packages the final dataset for TTS model fine-tuning.

## What gets exported

Only recordings that meet **all** of the following criteria are included:

1. `exclude_from_training == false`
2. `consent_tier` is `open` or `community` (not `restricted`)
3. `transcript` is not empty
4. The WAV file exists

Recordings that are present in the metadata for archival purposes but do not meet these criteria are preserved in the metadata file — they are never deleted, only excluded from the export.

## Output format

[LJSpeech format](https://keithito.com/LJ-Speech-Dataset/): pipe-separated CSV with columns `file_id|transcript|transcript`.


## 1. Configure

In [ ]:
import os

METADATA_CSV = "../metadata/metadata_template.csv"  # CARE-aligned metadata
WAVS_DIRS = [                                       # Source WAV directories (checked in order)
    "../test_data/wavs_export_audacity/",
    "../test_data/wavs_augmented/",
]

OUTPUT_DIR = "../output/"                   # Final export directory
OUTPUT_WAVS = os.path.join(OUTPUT_DIR, "wavs/")
OUTPUT_METADATA = os.path.join(OUTPUT_DIR, "metadata.csv")

os.makedirs(OUTPUT_WAVS, exist_ok=True)

def find_wav(file_id: str) -> str | None:
    """Find a WAV file across source directories."""
    for d in WAVS_DIRS:
        path = os.path.join(d, f"{file_id}.wav")
        if os.path.exists(path):
            return path
    return None

## 2. Load and Filter Metadata

In [ ]:
import pandas as pd

df = pd.read_csv(METADATA_CSV, dtype=str).fillna("")
print(f"Total rows in metadata: {len(df)}")

# Apply export filters
eligible = df[
    (df["exclude_from_training"].str.lower() != "true") &
    (df["consent_tier"].str.lower().isin(["open", "community"])) &
    (df["transcript"].str.strip() != "")
].copy()

excluded = df[~df.index.isin(eligible.index)]

print(f"Eligible for export: {len(eligible)}")
print(f"Excluded from export: {len(excluded)}")

if len(excluded) > 0:
    print("\nExcluded files:")
    for _, row in excluded.iterrows():
        reason = row.get('exclude_reason', '') or row.get('consent_tier', '')
        print(f"  {row['file_id']}: {reason}")

## 3. Validate WAV Files

In [ ]:
missing = []
for _, row in eligible.iterrows():
    if find_wav(row["file_id"]) is None:
        missing.append(row["file_id"])

if missing:
    print(f"WARNING: {len(missing)} WAV files not found:")
    for fid in missing:
        print(f"  {fid}.wav")
    print("These will be skipped in the export.")
    eligible = eligible[~eligible["file_id"].isin(missing)]
else:
    print(f"All {len(eligible)} WAV files found.")

## 4. Copy WAVs and Write Metadata

In [ ]:
from shutil import copyfile

ljspeech_lines = []

for _, row in eligible.iterrows():
    file_id = row["file_id"]
    transcript = row["transcript"].strip()
    
    src = find_wav(file_id)
    dst = os.path.join(OUTPUT_WAVS, f"{file_id}.wav")
    copyfile(src, dst)
    
    ljspeech_lines.append(f"{file_id}|{transcript}|{transcript}")

with open(OUTPUT_METADATA, "w", encoding="utf-8") as f:
    f.write("\n".join(ljspeech_lines) + "\n")

print(f"Exported {len(ljspeech_lines)} recordings to {OUTPUT_DIR}")
print(f"WAVs: {OUTPUT_WAVS}")
print(f"Metadata: {OUTPUT_METADATA}")

## 5. Final Summary

In [ ]:
print("=== Export Summary ===")
print(f"Total recordings in archive: {len(df)}")
print(f"Exported to training dataset: {len(eligible)}")
print(f"Preserved in archive only: {len(excluded)}")
print()
print("Consent tier breakdown of exported recordings:")
print(eligible["consent_tier"].value_counts().to_string())
print()
print("Preview of metadata.csv:")
with open(OUTPUT_METADATA) as f:
    for line in f.readlines()[:5]:
        print(" ", line.rstrip())

## What to do with the output

The `output/` directory is now in LJSpeech format:

```
output/
├── wavs/
│   ├── 001.wav
│   └── ...
└── metadata.csv
```

This can be used directly with:
- [Coqui TTS](https://github.com/coqui-ai/TTS) (`tts --model_name tts_models/en/ljspeech/...`)
- [Mozilla TTS](https://github.com/mozilla/TTS)
- [VITS](https://github.com/jaywalnut310/vits) and other LJSpeech-compatible trainers

**Reminder:** The consent tier (`open` vs `community`) determines how the trained model can be distributed. A model trained on `community` data belongs to the community.